# IoT 传感器数据分析 — pandas & numpy 实战演练

## 学习目标
- **NumPy**：数组操作、随机数生成、数学函数、条件筛选
- **pandas**：DataFrame 创建、清洗、分组聚合、滚动窗口、时间序列
- **Matplotlib + Seaborn**：趋势图、分布图、热力图、箱线图

### 数据场景
模拟一台工业设备的 IoT 传感器，每 **5 秒** 采集一条数据，持续 **24 小时**，包含：
- **振动 (vibration)**：mm/s
- **温度 (temperature)**：°C
- **电流 (current)**：A

数据中人工加入了**缺失值**和**异常尖峰**，用于练习数据清洗和异常检测。

In [ ]:
# ============================================================
# 1. 环境准备
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设定随机种子，结果可复现
np.random.seed(42)

# 绘图设置
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('环境准备完成')

---
## 2. 用 NumPy 生成模拟传感器数据

### 知识点
- `np.linspace` / `np.arange`：生成时间轴
- `np.random.normal`：正态分布随机噪声
- `np.sin`：模拟温度昼夜周期
- `np.where`：条件插入异常值
- `np.concatenate`：合并数组

In [ ]:
# 参数设置
INTERVAL_SEC = 5          # 采集间隔（秒）
HOURS = 24                # 采集时长（小时）
N = HOURS * 3600 // INTERVAL_SEC   # 总样本数

print(f'总样本数: {N}')

# ---- 时间轴 ----
timestamps = pd.date_range(
    start='2025-06-01 00:00:00',
    periods=N,
    freq=f'{INTERVAL_SEC}s'
)
print(f'时间范围: {timestamps[0]} ~ {timestamps[-1]}')

In [ ]:
# ---- 1. 振动信号 (vibration) ----
# 基准 5 mm/s + 随机噪声 (均值0, 标准差1.2)，再插入 10 个异常高峰
vibration = 5.0 + np.random.normal(0, 1.2, N)

# 用 np.random.choice 随机选 10 个位置，插入异常值 (15~25 mm/s)
anomaly_idx = np.random.choice(N, size=10, replace=False)
vibration[anomaly_idx] += np.random.uniform(10, 20, size=10)

print(f'振动数据: 均值={vibration.mean():.2f}, 标准差={vibration.std():.2f}')
print(f'异常位置(前5个): {anomaly_idx[:5]}')

In [ ]:
# ---- 2. 温度信号 (temperature) ----
# 昼夜周期: 白天高 (~45°C), 夜间低 (~25°C), 用 sin 模拟
# 周期 = 24小时, 峰值在 14:00 左右
hours = np.arange(N) * INTERVAL_SEC / 3600  # 转为小时
temp_base = 35 + 10 * np.sin(2 * np.pi * (hours - 8) / 24)
temperature = temp_base + np.random.normal(0, 1.5, N)

print(f'温度数据: 均值={temperature.mean():.2f}°C, 范围=[{temperature.min():.1f}, {temperature.max():.1f}]°C')

In [ ]:
# ---- 3. 电流信号 (current) ----
# 基准 10A，随振动和温度略有波动，加噪声
current = (10.0
           + 0.3 * (vibration - 5.0)        # 振动越大电流略增
           + 0.05 * (temperature - 35)      # 温度越高电流略增
           + np.random.normal(0, 0.8, N))
current = np.clip(current, 5, 20)  # 限制合理范围 [5, 20]A

print(f'电流数据: 均值={current.mean():.2f}A, 范围=[{current.min():.1f}, {current.max():.1f}]A')

In [ ]:
# ---- 4. 人工加入缺失值 (NaN) ----
# 随机选 50 个位置设为 NaN，模拟传感器丢包
nan_idx_vib = np.random.choice(N, size=50, replace=False)
nan_idx_temp = np.random.choice(N, size=30, replace=False)
nan_idx_cur = np.random.choice(N, size=20, replace=False)

vibration[nan_idx_vib] = np.nan
temperature[nan_idx_temp] = np.nan
current[nan_idx_cur] = np.nan

print(f'缺失值: 振动={len(nan_idx_vib)}, 温度={len(nan_idx_temp)}, 电流={len(nan_idx_cur)}')

---
## 3. 创建 DataFrame 与探索性分析

### 知识点
- `pd.DataFrame`：从 dict 创建
- `df.info()`：列类型、非空值数
- `df.describe()`：统计摘要
- `df.head()`, `df.tail()`：查看首尾
- `df.isna().sum()`：缺失值统计

In [ ]:
df = pd.DataFrame({
    'timestamp': timestamps,
    'vibration': vibration,
    'temperature': temperature,
    'current': current
})

print(f'DataFrame 形状: {df.shape}')
print(f'内存占用: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 缺失值统计
missing = df.isna().sum()
missing_pct = df.isna().mean() * 100
pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct.round(2)})

---
## 4. 数据清洗

### 知识点
- `df.interpolate()`：线性插值填充缺失值
- `scipy.stats.zscore`：Z-score 异常检测
- `np.abs()`：绝对值
- 布尔索引筛选：`df[df['vibration'] > threshold]`

In [ ]:
# 4.1 时间序列插值填充缺失值
# 将 timestamp 设为索引，然后按时间线性插值
df = df.set_index('timestamp')
df = df.interpolate(method='time')

# 检查是否还有缺失值
print(f'插值后缺失值数量: {df.isna().sum().sum()}')
df.head()

In [ ]:
# 4.2 Z-score 异常检测（纯 numpy 实现）
def zscore(arr):
    return (arr - np.nanmean(arr)) / np.nanstd(arr)

z_vibration = np.abs(zscore(df['vibration'].values))
z_temperature = np.abs(zscore(df['temperature'].values))
z_current = np.abs(zscore(df['current'].values))

THRESHOLD_Z = 3
df['vibration_anomaly'] = z_vibration > THRESHOLD_Z
df['temperature_anomaly'] = z_temperature > THRESHOLD_Z
df['current_anomaly'] = z_current > THRESHOLD_Z

print(f'振动异常点: {df["vibration_anomaly"].sum()}')
print(f'温度异常点: {df["temperature_anomaly"].sum()}')
print(f'电流异常点: {df["current_anomaly"].sum()}')

In [ ]:
# 查看具体哪些行被标记为异常
df[df['vibration_anomaly']].head(10)

---
## 5. 特征工程 — 滚动窗口与滞后特征

### 知识点
- `df.rolling(window)`：滚动窗口
- `df.shift(periods)`：滞后/超前移位
- 新特征：滚动均值、滚动标准差、传感器比值

In [ ]:
# 5.1 滚动窗口统计（窗口 = 1 小时 = 720 个样本）
WINDOW = 720  # 5秒 * 720 = 3600秒 = 1小时

df['vibration_ma'] = df['vibration'].rolling(window=WINDOW).mean()   # 移动平均
df['vibration_std'] = df['vibration'].rolling(window=WINDOW).std()   # 移动标准差
df['temperature_ma'] = df['temperature'].rolling(window=WINDOW).mean()

df[['vibration', 'vibration_ma', 'vibration_std']].head(10)

In [ ]:
# 5.2 滞后特征：上一时刻的值
df['vibration_lag1'] = df['vibration'].shift(1)       # t-1
df['vibration_lag2'] = df['vibration'].shift(2)       # t-2
df['temp_change'] = df['temperature'].diff()           # 一阶差分

# 5.3 传感器比值（工程特征）
df['vib_temp_ratio'] = df['vibration'] / df['temperature']
df['vib_current_ratio'] = df['vibration'] / df['current']

df[['vibration', 'vibration_lag1', 'vibration_lag2', 'temp_change', 'vib_temp_ratio']].head(10)

---
## 6. 分组聚合 — 按时间窗口统计

### 知识点
- `df.resample(rule)`：时间维度重采样
- `df.groupby()`：分组
- `df.agg()` / `aggregate()`：多聚合函数
- `pd.Grouper`：按时间频率分组

In [ ]:
# 6.1 按小时重采样，聚合统计
hourly = df.resample('1h').agg({
    'vibration': ['mean', 'std', 'max', 'min'],
    'temperature': ['mean', 'std', 'max', 'min'],
    'current': ['mean', 'std', 'max', 'min']
})

# 拍平多层列名
hourly.columns = ['_'.join(col).strip() for col in hourly.columns.values]
hourly.head()

In [ ]:
# 6.2 按班次分组（早班 6-14点，中班 14-22点，夜班 22-6点）
def shift_label(h):
    if 6 <= h < 14:
        return '早班 (6-14)'
    elif 14 <= h < 22:
        return '中班 (14-22)'
    else:
        return '夜班 (22-6)'

df['hour'] = df.index.hour
df['shift'] = df['hour'].map(shift_label)

shift_stats = df.groupby('shift').agg({
    'vibration': ['mean', 'std', 'max'],
    'temperature': ['mean', 'std', 'max'],
    'current': ['mean', 'std', 'max']
}).round(2)

shift_stats

---
## 7. 关联分析

### 知识点
- `df.corr()`：皮尔逊相关系数矩阵
- `sns.heatmap()`：热力图可视化
- `sns.pairplot()`：成对关系图

In [ ]:
# 7.1 相关系数矩阵
corr_cols = ['vibration', 'temperature', 'current']
corr_matrix = df[corr_cols].corr()
corr_matrix.round(3)

In [ ]:
# 7.2 热力图
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', vmin=-1, vmax=1,
            linewidths=1, square=True)
plt.title('传感器相关性热力图', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 成对关系图（采样 2000 点以免太密）
sample_df = df[corr_cols].dropna().sample(2000, random_state=42)
sns.pairplot(sample_df, diag_kind='kde', corner=True)
plt.suptitle('传感器成对关系图', y=1.02, fontsize=14)
plt.show()

---
## 8. 异常检测实战

### 知识点
- **3σ 法则**：超出均值±3倍标准差
- **IQR 方法**：超出 Q1-1.5*IQR 或 Q3+1.5*IQR
- `np.where()`：条件筛选
- `df.quantile()`：分位数
- 两种方法结果对比

In [ ]:
# 8.1 3σ 法检测振动异常
vib_mean = df['vibration'].mean()
vib_std = df['vibration'].std()
upper_3sigma = vib_mean + 3 * vib_std
lower_3sigma = vib_mean - 3 * vib_std

df['anomaly_3sigma'] = (df['vibration'] > upper_3sigma) | (df['vibration'] < lower_3sigma)
print(f'3σ 法检测到异常: {df["anomaly_3sigma"].sum()} 个')
print(f'  上限={upper_3sigma:.2f}, 下限={lower_3sigma:.2f}')

In [ ]:
# 8.2 IQR 法检测振动异常
Q1 = df['vibration'].quantile(0.25)
Q3 = df['vibration'].quantile(0.75)
IQR = Q3 - Q1
upper_iqr = Q3 + 1.5 * IQR
lower_iqr = Q1 - 1.5 * IQR

df['anomaly_iqr'] = (df['vibration'] > upper_iqr) | (df['vibration'] < lower_iqr)
print(f'IQR 法检测到异常: {df["anomaly_iqr"].sum()} 个')
print(f'  上限={upper_iqr:.2f}, 下限={lower_iqr:.2f}')

In [ ]:
# 8.3 两种方法结果对比
inter = (df['anomaly_3sigma'] & df['anomaly_iqr']).sum()
comparison = pd.DataFrame({
    '方法': ['3σ 法则', 'IQR 方法'],
    '异常数量': [df['anomaly_3sigma'].sum(), df['anomaly_iqr'].sum()],
    '异常比例(%)': [
        round(df['anomaly_3sigma'].mean() * 100, 3),
        round(df['anomaly_iqr'].mean() * 100, 3)
    ]
})
print(f'两种方法共同检测到的异常: {inter}')
comparison

---
## 9. 数据可视化

### 知识点
- `plt.subplots()`：多子图
- `sns.lineplot()` / `sns.scatterplot()`：折线图/散点图
- `sns.kdeplot()` / `sns.histplot()`：分布图
- `sns.boxplot()`：箱线图
- 异常点高亮标注

In [ ]:
# 9.1 传感器趋势图（取前 2 小时数据较清晰）
plot_df = df.iloc[:1440]  # 2小时 = 1440个样本

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# 振动
axes[0].plot(plot_df.index, plot_df['vibration'], color='#E74C3C', alpha=0.6, linewidth=0.8)
axes[0].plot(plot_df.index, plot_df['vibration_ma'], color='#2C3E50', linewidth=1.5, label='1h 移动平均')
anom = plot_df[plot_df['vibration_anomaly']]
axes[0].scatter(anom.index, anom['vibration'], color='red', s=30, label='异常', zorder=5)
axes[0].set_ylabel('振动 (mm/s)')
axes[0].legend()
axes[0].set_title('振动传感器趋势')

# 温度
axes[1].plot(plot_df.index, plot_df['temperature'], color='#E67E22', alpha=0.7, linewidth=0.8)
axes[1].plot(plot_df.index, plot_df['temperature_ma'], color='#2C3E50', linewidth=1.5, label='1h 移动平均')
axes[1].set_ylabel('温度 (°C)')
axes[1].legend()
axes[1].set_title('温度传感器趋势')

# 电流
axes[2].plot(plot_df.index, plot_df['current'], color='#3498DB', alpha=0.6, linewidth=0.8)
axes[2].set_ylabel('电流 (A)')
axes[2].set_title('电流传感器趋势')

plt.xlabel('时间')
plt.tight_layout()
plt.show()

In [ ]:
# 9.2 分布图：直方图 + KDE
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['vibration'], kde=True, bins=60, ax=axes[0], color='#E74C3C')
axes[0].axvline(vib_mean, color='black', linestyle='--', label=f'均值={vib_mean:.1f}')
axes[0].axvline(upper_3sigma, color='red', linestyle=':', label=f'3σ上限={upper_3sigma:.1f}')
axes[0].legend(fontsize=9)
axes[0].set_title('振动分布')

sns.histplot(df['temperature'], kde=True, bins=60, ax=axes[1], color='#E67E22')
axes[1].set_title('温度分布')

sns.histplot(df['current'], kde=True, bins=60, ax=axes[2], color='#3498DB')
axes[2].set_title('电流分布')

plt.tight_layout()
plt.show()

In [ ]:
# 9.3 按班次箱线图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=df, x='shift', y='vibration', ax=axes[0],
            palette='Set2', order=['早班 (6-14)', '中班 (14-22)', '夜班 (22-6)'])
axes[0].set_title('各班组振动分布')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='shift', y='temperature', ax=axes[1],
            palette='Set2', order=['早班 (6-14)', '中班 (14-22)', '夜班 (22-6)'])
axes[1].set_title('各班组温度分布')
axes[1].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='shift', y='current', ax=axes[2],
            palette='Set2', order=['早班 (6-14)', '中班 (14-22)', '夜班 (22-6)'])
axes[2].set_title('各班组电流分布')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## 10. 综合实战：设备报警规则

模拟真实 IoT 场景：当传感器同时满足多个条件时触发报警。

### 报警规则
- 振动 > 8 mm/s **且** 温度 > 40°C → **严重告警**
- 振动 > 8 mm/s **或** 电流 > 15 A → **预警**

### 知识点
- 多条件组合筛选：`&`, `|`
- `np.select()`：多条件多结果的向量化操作
- `df.groupby().size()`：按类别统计

In [ ]:
conditions = [
    (df['vibration'] > 8) & (df['temperature'] > 40),   # 严重告警
    (df['vibration'] > 8) | (df['current'] > 15)         # 预警
]
choices = ['严重告警', '预警']

df['alert_level'] = np.select(conditions, choices, default='正常')

# 统计各类别数量
alert_counts = df['alert_level'].value_counts()
alert_counts

In [ ]:
# 可视化报警分布
colors = {'正常': '#2ECC71', '预警': '#F1C40F', '严重告警': '#E74C3C'}
plt.figure(figsize=(8, 5))
bars = plt.bar(alert_counts.index, alert_counts.values,
               color=[colors[c] for c in alert_counts.index],
               edgecolor='black', linewidth=1.2)
plt.title('设备报警分布', fontsize=14)
plt.ylabel('样本数')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=12)
plt.show()

In [ ]:
# 查看几次严重告警的详细数据
df[df['alert_level'] == '严重告警'][['vibration', 'temperature', 'current']].head(10)

---
## 总结

### 本 Notebook 涉及的 pandas & numpy 知识点

| 类别 | 函数/方法 |
|------|-----------|
| **NumPy 基础** | `np.array`, `np.random.normal`, `np.random.choice`, `np.sin`, `np.abs`, `np.where`, `np.select`, `np.clip` |
| **NumPy 统计** | `np.mean`, `np.std`, `np.min`, `np.max` |
| **pandas 创建** | `pd.DataFrame`, `pd.date_range`, `pd.Series` |
| **pandas 查看** | `df.head()`, `df.info()`, `df.describe()`, `df.isna()` |
| **pandas 清洗** | `df.interpolate()`, `df.dropna()`, `df.fillna()` |
| **pandas 特征** | `df.rolling()`, `df.shift()`, `df.diff()` |
| **pandas 分组** | `df.groupby()`, `df.resample()`, `df.agg()`, `df.value_counts()` |
| **pandas 筛选** | 布尔索引, `df.loc[]`, `df.iloc[]`, `df.query()` |
| **可视化** | `plt.plot()`, `sns.histplot()`, `sns.boxplot()`, `sns.heatmap()`, `sns.pairplot()` |

### 扩展思考
- 如何用 `scikit-learn` 的 Isolation Forest 做无监督异常检测？
- 如何用 `statsmodels` 做时序分解（趋势+季节+残差）？
- 如何用 TensorFlow/PyTorch 搭建 LSTM 预测传感器趋势？